# Surgery Phase Detection — MVOR Data Exploration

This notebook explores the MVOR (Multi-View Operating Room) dataset:
- Scene composition (people, roles)
- Temporal patterns across recording days
- Phase label distributions derived from scene annotations
- Multi-view visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

from surgery_phase_detection import MVOR_PHASE_NAMES, NUM_CLASSES
from surgery_phase_detection.data.phase_labeler import MVORPhaseLabeler
from surgery_phase_detection.utils.visualization import PHASE_COLORS

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 1. Load MVOR Annotations

In [ ]:
ANNOTATION_PATH = '../data/mvor/annotations/camma_mvor_2018.json'
DATASET_ROOT = '../data/mvor/'

if os.path.exists(ANNOTATION_PATH):
    with open(ANNOTATION_PATH) as f:
        mvor_data = json.load(f)
    print(f'Top-level keys: {list(mvor_data.keys())}')
    print(f'Images: {len(mvor_data["images"])}')
    print(f'Annotations: {len(mvor_data["annotations"])}')
    print(f'Multi-view frames: {len(mvor_data["multiview_images"])}')
    print(f'3D annotations: {len(mvor_data["annotations3D"])}')
else:
    print(f'Annotation file not found at {ANNOTATION_PATH}')
    print('Run: python scripts/download_mvor.py --output_dir data/mvor --annotations_only')
    mvor_data = None

## 2. Dataset Overview

In [ ]:
if mvor_data:
    # Person role distribution
    role_counts = Counter(ann['person_role'] for ann in mvor_data['annotations'])
    print('Person role distribution:')
    for role, count in role_counts.items():
        print(f'  {role}: {count}')
    
    # Day distribution
    day_counts = Counter(img['day_id'] for img in mvor_data['images'])
    cam_counts = Counter(img['cam_id'] for img in mvor_data['images'])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    days = sorted(day_counts.keys())
    ax1.bar([f'Day {d}' for d in days], [day_counts[d] for d in days],
            color=['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'])
    ax1.set_ylabel('Number of Images')
    ax1.set_title('Images per Recording Day')
    
    cams = sorted(cam_counts.keys())
    ax2.bar([f'Camera {c}' for c in cams], [cam_counts[c] for c in cams],
            color=['#9B59B6', '#1ABC9C', '#E67E22'])
    ax2.set_ylabel('Number of Images')
    ax2.set_title('Images per Camera')
    
    plt.tight_layout()
    plt.show()

## 3. Scene Occupancy Over Time

Analyze how many people (and their roles) are present across the day.

In [ ]:
if mvor_data:
    # Build annotation index
    anns_by_image = defaultdict(list)
    for ann in mvor_data['annotations']:
        anns_by_image[ann['image_id']].append(ann)
    
    # Analyze per-multiview-frame occupancy
    records = []
    for mv in mvor_data['multiview_images']:
        img0 = mv['images'][0]
        day_id = img0['day_id']
        timestamp = img0['date_captured']
        
        # Aggregate people across all views
        all_anns = []
        for img_info in mv['images']:
            all_anns.extend(anns_by_image.get(img_info['id'], []))
        
        # Deduplicate by person_id
        unique_persons = {}
        for ann in all_anns:
            unique_persons[ann['person_id']] = ann
        
        n_people = len(unique_persons)
        n_clinicians = sum(1 for p in unique_persons.values() if p['person_role'] == 'clinician')
        n_patients = sum(1 for p in unique_persons.values() if p['person_role'] == 'patient')
        
        records.append({
            'day_id': day_id,
            'timestamp': timestamp,
            'n_people': n_people,
            'n_clinicians': n_clinicians,
            'n_patients': n_patients,
        })
    
    occ_df = pd.DataFrame(records)
    occ_df['timestamp'] = pd.to_datetime(occ_df['timestamp'])
    occ_df = occ_df.sort_values(['day_id', 'timestamp'])
    
    # Plot per day
    for day_id in sorted(occ_df['day_id'].unique()):
        day_data = occ_df[occ_df['day_id'] == day_id].copy()
        day_data['time_min'] = (day_data['timestamp'] - day_data['timestamp'].min()).dt.total_seconds() / 60
        
        fig, ax = plt.subplots(figsize=(14, 3))
        ax.fill_between(day_data['time_min'], day_data['n_clinicians'], alpha=0.7, 
                       label='Clinicians', color='#3498DB')
        ax.fill_between(day_data['time_min'], day_data['n_clinicians'],
                       day_data['n_clinicians'] + day_data['n_patients'],
                       alpha=0.7, label='Patients', color='#E74C3C')
        ax.set_xlabel('Time (minutes from start)')
        ax.set_ylabel('People Count')
        ax.set_title(f'Day {day_id} — OR Occupancy ({len(day_data)} frames)')
        ax.legend()
        ax.set_ylim(0, day_data['n_people'].max() + 1)
        plt.tight_layout()
        plt.show()

## 4. Derived Phase Labels

Apply the phase labeler to derive surgical phases from scene context.

In [ ]:
if mvor_data:
    labeler = MVORPhaseLabeler()
    mv_phases = labeler.label_all_frames(mvor_data['multiview_images'], anns_by_image)
    
    # Phase distribution
    phase_counts = Counter(mv_phases.values())
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    phases = sorted(phase_counts.keys())
    counts = [phase_counts[p] for p in phases]
    names = [MVOR_PHASE_NAMES[p] for p in phases]
    colors = [PHASE_COLORS[p] for p in phases]
    
    ax1.bar(names, counts, color=colors, edgecolor='black', linewidth=0.5)
    ax1.set_ylabel('Frame Count')
    ax1.set_title('Derived Phase Distribution')
    for i, (n, c) in enumerate(zip(names, counts)):
        ax1.text(i, c + 2, str(c), ha='center', fontsize=10)
    
    ax2.pie(counts, labels=names, colors=colors, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Phase Proportions')
    
    plt.tight_layout()
    plt.show()

## 5. Phase Timeline per Day

Visualize the temporal structure of derived phases.

In [ ]:
if mvor_data:
    # Build per-day phase timelines
    day_phases = defaultdict(list)
    for mv in mvor_data['multiview_images']:
        mv_id = mv['id']
        day_id = mv['images'][0]['day_id']
        timestamp = mv['images'][0]['date_captured']
        phase = mv_phases.get(mv_id, 0)
        day_phases[day_id].append((timestamp, phase))
    
    n_days = len(day_phases)
    fig, axes = plt.subplots(n_days, 1, figsize=(16, 2.5 * n_days))
    if n_days == 1:
        axes = [axes]
    
    for idx, day_id in enumerate(sorted(day_phases.keys())):
        frames = sorted(day_phases[day_id], key=lambda x: x[0])
        phases = np.array([f[1] for f in frames])
        time_min = np.arange(len(phases))
        
        ax = axes[idx]
        for c in range(NUM_CLASSES):
            mask = phases == c
            if np.any(mask):
                ax.fill_between(time_min, 0, 1, where=mask,
                              color=PHASE_COLORS[c], alpha=0.8,
                              transform=ax.get_xaxis_transform())
        ax.set_ylabel(f'Day {day_id}', fontsize=10)
        ax.set_yticks([])
        ax.set_xlim(0, len(phases) - 1)
    
    axes[-1].set_xlabel('Frame Index')
    
    handles = [plt.Rectangle((0,0),1,1, fc=PHASE_COLORS[i]) for i in range(NUM_CLASSES)]
    fig.legend(handles, MVOR_PHASE_NAMES, loc='center right',
              bbox_to_anchor=(1.15, 0.5), fontsize=9)
    
    fig.suptitle('Derived Phase Timelines per Day', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Person Pose Analysis

Analyze keypoint annotations and pose patterns.

In [ ]:
if mvor_data:
    # Keypoint visibility stats
    kp_names = mvor_data['categories'][0]['keypoints']
    kp_visible = np.zeros(len(kp_names))
    total_with_kps = 0
    
    for ann in mvor_data['annotations']:
        if not ann.get('only_bbox', 0):
            kps = np.array(ann['keypoints']).reshape(-1, 3)
            for i in range(len(kp_names)):
                if i < len(kps) and kps[i, 2] > 0:
                    kp_visible[i] += 1
            total_with_kps += 1
    
    kp_percent = kp_visible / max(total_with_kps, 1) * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(kp_names, kp_percent, color='#3498DB', edgecolor='black', linewidth=0.5)
    ax.set_ylabel('Visibility (%)')
    ax.set_title(f'Keypoint Visibility ({total_with_kps} annotations with keypoints)')
    ax.set_ylim(0, 100)
    plt.xticks(rotation=45, ha='right')
    for bar, pct in zip(bars, kp_percent):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'{pct:.0f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 7. Train/Val/Test Split Overview

In [ ]:
if mvor_data:
    split_config = {
        'Train': [2, 3],
        'Validation': [4],
        'Test': [1],
    }
    
    print('Dataset Split by Recording Day:')
    print('-' * 50)
    
    for split_name, days in split_config.items():
        n_frames = sum(1 for mv in mvor_data['multiview_images']
                      if mv['images'][0]['day_id'] in days)
        split_phases = [mv_phases[mv['id']] for mv in mvor_data['multiview_images']
                       if mv['images'][0]['day_id'] in days]
        phase_dist = Counter(split_phases)
        
        print(f'\n{split_name} (Days {days}): {n_frames} multi-view frames')
        for p in sorted(phase_dist.keys()):
            name = MVOR_PHASE_NAMES[p]
            print(f'  Phase {p} ({name}): {phase_dist[p]} frames')